#  FinNavigator: Qwen3-VL Fine-Tuning
This notebook handles  fine-tuning of the Qwen3-VL-4B model using Unsloth for 2x speed and TrackIO for experiment tracking. It merges baseline training data with the  generated synthetic career strategies.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install trackio datasets transformers torch

: 

In [ ]:
import os
import torch
import json
from datasets import load_dataset
from transformers import TrainingArguments, TrainerCallback
from trl import SFTTrainer
from unsloth import FastLanguageModel
import trackio

#  Initialize TrackIO
run = trackio.init(project="FinNavigator-LLM", run_name="Qwen-3-VL-4B-Instruct-SFT")

class TrackIOCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            run.log(logs, step=state.global_step)

## Data Preparation
We merge `finnav_train.jsonl` and `synthetic_career_data.jsonl` to provide the agent with both foundational knowledge and specific career strategies.

In [ ]:
def format_prompts(examples):
    texts = []
    for instruction, input_text, output in zip(examples["instruction"], examples["input"], examples["output"]):
        prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input_text}

### Response:
{output}"""
        texts.append(prompt)
    return {"text": texts}

data_files = [
    "finnav_train.jsonl",
    "synthetic_career_data.jsonl"
]

existing_files = [f for f in data_files if os.path.exists(f)]
dataset = load_dataset("json", data_files=existing_files, split="train")
dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = dataset["train"].map(format_prompts, batched=True)
eval_dataset = dataset["test"].map(format_prompts, batched=True)

print(f"📊 Loaded {len(train_dataset)} Train, {len(eval_dataset)} Eval samples.")

##  Model Setup
Loading the Qwen3-VL 4B model in 4-bit quantization for maximum efficiency.

In [ ]:
max_seq_length = 2048
dtype = None 
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-VL-4B-Instruct-bnb-4bit", 
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",],
    lora_alpha=16,
    lora_dropout=0, 
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## Training
Executing the fine-tuning run.

In [ ]:
args = TrainingArguments(
    output_dir="outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    evaluation_strategy="steps",
    eval_steps=10,
    optim="adamw_8bit",
    seed=3407,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=args,
    callbacks=[TrackIOCallback()]
)

run.log_params({"model": "Qwen3-VL-4B", "epochs": 2, "lora_r": 16})
trainer.train()
run.finish()

##  Save & Push
Saving the adapters and pushing to the Hugging Face Hub.

In [ ]:
model.save_pretrained("finnav_qwen3-VL4b_lora")
tokenizer.save_pretrained("finnav_qwen3.5_4b_lora")

# Replace with your HF token or use notebook_login()
model.push_to_hub("MOH749/finnav_qwen3.5_4b_lora")
tokenizer.push_to_hub("MOH749/finnav_qwen3.5_4b_lora")